In [1]:
 #Importing the libraries.
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

In [2]:
# Loading the structurally cleaned dataset.
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "application_train_structural_cleaned.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (307511, 105)


In [3]:
#Separating the features and target variable.
X=df.drop(columns=["TARGET", "SK_ID_CURR"])
y=df["TARGET"]

In [4]:
#Train-test Split.
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#Verifying the shape of the train and test sets.
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (246008, 103)
X_test shape: (61503, 103)
y_train shape: (246008,)
y_test shape: (61503,)


In [5]:
#Identify Column Types
numeric_features=X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features=X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Numerical Features: {len(numeric_features)}")
print(f"Categorical Features: {len(categorical_features)}")

Numerical Features: 88
Categorical Features: 15


C:\Users\unive\AppData\Local\Temp\ipykernel_25788\392342828.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features=X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


In [6]:
# Building the leakage-safe preprocessing pipeline.
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),
                                      ("scaler", StandardScaler())])

categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                                          ("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features),
                                               ("cat", categorical_transformer, categorical_features)])

# Fit preprocessing parameters using training data only.
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the fitted transformations to the test data.
X_test_processed = preprocessor.transform(X_test)

print("Preprocessor fitted on training data only.")
print(f"Training observations: {X_train.shape[0]:,}")
print(f"Testing observations: {X_test.shape[0]:,}")
print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed testing shape: {X_test_processed.shape}")

Preprocessor fitted on training data only.
Training observations: 246,008
Testing observations: 61,503
Processed training shape: (246008, 224)
Processed testing shape: (61503, 224)


# Machine Learning Preprocessing Pipeline

## Numerical Features
- Missing Values -> Median
- Scaling -> StandardScaler

## Categorical Features
- Missing Values -> Most Frequent Value
- Encoding -> One-Hot Encoding

## Why use a Preprocessing Pipeline>?

The preprocessing transformations are implemented using the scikit-learn pipelines and a ColumnTransformer. This ensures that imputation, scaling and categorical encoding are fitted using the training data and then consistently applied to the test data.

This prevents information from the test set from influencing the preprocessing parameters and reduces the risk of data leakage.

In [7]:
# Transformed Dimensions of the processed dataset.
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("Original feature count:", X_train.shape[1])
print("Processed feature count:", X_train_processed.shape[1])

Processed training shape: (246008, 224)
Processed testing shape: (61503, 224)
Original feature count: 103
Processed feature count: 224


The number of model-ready features may exceed the original number of variables because categorical variables are expanded through one-hot encoding.